# Tutorial 07: Empirical Privacy Auditing

**Level**: Advanced  
**Duration**: 45-60 minutes  
**Prerequisites**: Tutorials 01-03, basic understanding of DP guarantees

## Overview

While privacy accounting gives us *theoretical* upper bounds on privacy loss, **privacy auditing** provides *empirical* lower bounds by actually attacking the trained model.

In this tutorial, you'll learn:

1. **Why audit?** - The gap between theory and practice
2. **Membership inference** - How attacks reveal privacy leakage
3. **One-run auditing** - End-to-end workflow with `opaque.auditing`
4. **Epsilon estimation** - Converting attack success to privacy bounds
5. **Attack metrics** - AUROC, TPR@FPR, and accuracy
6. **Bootstrap confidence intervals** - Quantifying uncertainty

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import opaque.auditing as auditing
from opaque.auditing import AuditResult, BootstrapParams

np.random.seed(42)

## 1. Why Audit?

Privacy accounting tells us the *worst-case* privacy loss. But in practice:

- The actual privacy loss may be much lower
- Implementation bugs can cause higher-than-expected leakage
- We want to validate that our DP implementation is correct

**The goal**: Find the largest epsilon that an attacker can empirically demonstrate.

```
audited_epsilon <= true_epsilon <= theoretical_epsilon
```

If `audited_epsilon > theoretical_epsilon`, there's likely a bug!

## 2. Membership Inference Attacks

The basic idea:
1. Train a model on dataset D
2. For each example x, compute a "membership score" indicating how likely x was in D
3. Use these scores to distinguish training members from non-members

Common scoring functions:
- **Loss**: Training members typically have lower loss
- **Confidence**: Training members typically have higher prediction confidence
- **LiRA**: Likelihood ratio comparing "trained with x" vs "trained without x"

Let's simulate some attack scores:

In [ ]:
# Simulate membership inference attack scores
# In practice, these come from actually training a model and running an attack

n_canaries = 500  # Number of canary examples

# Scenario 1: Good DP (small separation between in/out)
in_scores_good_dp = np.random.normal(loc=0.55, scale=0.3, size=n_canaries)
out_scores_good_dp = np.random.normal(loc=0.45, scale=0.3, size=n_canaries)

# Scenario 2: Weak DP (larger separation)
in_scores_weak_dp = np.random.normal(loc=0.7, scale=0.25, size=n_canaries)
out_scores_weak_dp = np.random.normal(loc=0.3, scale=0.25, size=n_canaries)

# Scenario 3: No DP (very large separation - privacy breach)
in_scores_no_dp = np.random.normal(loc=0.9, scale=0.1, size=n_canaries)
out_scores_no_dp = np.random.normal(loc=0.1, scale=0.1, size=n_canaries)

In [ ]:
# Visualize the score distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

scenarios = [
    ("Good DP (ε≈3)", in_scores_good_dp, out_scores_good_dp),
    ("Weak DP (ε≈8)", in_scores_weak_dp, out_scores_weak_dp),
    ("No DP (ε→∞)", in_scores_no_dp, out_scores_no_dp),
]

for ax, (title, in_s, out_s) in zip(axes, scenarios):
    ax.hist(in_s, bins=30, alpha=0.6, label="In (training)", density=True)
    ax.hist(out_s, bins=30, alpha=0.6, label="Out (test)", density=True)
    ax.set_xlabel("Membership Score")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

## 3. Epsilon Estimation

The key insight: if a mechanism is (epsilon, delta)-DP, then for any threshold t:

$$\text{TPR}(t) \leq e^\epsilon \cdot \text{FPR}(t) + \delta$$

Rearranging:

$$\epsilon \geq \log\left(\frac{\text{TPR}(t) - \delta}{\text{FPR}(t)}\right)$$

`AuditResult` provides two methods:

- **Clopper-Pearson** (`method="clopper_pearson"`): Conservative binomial CI, works for any in/out split
- **One-run** (`method="one_run"`): Tighter, assumes coin-flip setup (Steinke et al. 2023)

The unified `epsilon_at()` selects the method automatically based on how the result was created.

In [ ]:
# Create an AuditResult directly from scores
result = AuditResult(in_scores_weak_dp, out_scores_weak_dp)

# Method 1: Clopper-Pearson (default when constructed directly)
eps_cp = result.epsilon_at(delta=1e-5, significance=0.05)

# Method 2: One-run (must specify explicitly here)
eps_one = result.epsilon_at(delta=1e-5, significance=0.05, method="one_run")

# Or call methods directly:
eps_cp_direct = result.epsilon_clopper_pearson(significance=0.05, delta=1e-5)
eps_one_direct = result.epsilon_one_run(significance=0.05, delta=1e-5)

print("Epsilon Estimates (Weak DP scenario):")
print(f"  Clopper-Pearson: {eps_cp:.2f}")
print(f"  One-run:         {eps_one:.2f}")

In [ ]:
# Compare across all scenarios
print("Epsilon Estimates by Scenario (Clopper-Pearson):")
print("-" * 40)

for name, in_s, out_s in scenarios:
    r = AuditResult(in_s, out_s)
    eps = r.epsilon_at(delta=1e-5, significance=0.05)
    print(f"{name:20s}: epsilon >= {eps:.2f}")

## 4. Attack Utility Metrics

Beyond epsilon, these methods help understand attack strength:

In [ ]:
# AUROC: Area Under ROC Curve
# 0.5 = random guessing, 1.0 = perfect attack

print("Attack AUROC by Scenario:")
print("-" * 40)

for name, in_s, out_s in scenarios:
    r = AuditResult(in_s, out_s)
    print(f"{name:20s}: AUROC = {r.auroc():.3f}")

In [ ]:
# TPR at low FPR: Important for real-world attacks where false positives are costly

result = AuditResult(in_scores_weak_dp, out_scores_weak_dp)
fprs = [0.001, 0.01, 0.1]

print("TPR at various FPR thresholds (Weak DP scenario):")
print("-" * 40)

for fpr_val in fprs:
    tpr_val = result.tpr_at_fpr(fpr=fpr_val)
    print(f"  FPR = {fpr_val:.1%}: TPR = {tpr_val:.3f}")

In [ ]:
# Max accuracy: Best achievable classification accuracy

print("\nMax Accuracy by Scenario:")
print("-" * 40)

for name, in_s, out_s in scenarios:
    r = AuditResult(in_s, out_s)
    print(f"{name:20s}: Accuracy = {r.max_accuracy():.1%}")

## 5. The `summary()` Method

Get all metrics at once:

In [ ]:
# Comprehensive summary in one call
result = AuditResult(in_scores_weak_dp, out_scores_weak_dp)

print(result.summary(significance=0.05, delta=1e-5))
print()
print(repr(result))

In [ ]:
# Individual metrics are methods, not tuple unpacking
result = AuditResult(in_scores_weak_dp, out_scores_weak_dp)

eps = result.epsilon_at(delta=1e-5)
auroc = result.auroc()
tpr = result.tpr_at_fpr(fpr=0.01)
acc = result.max_accuracy()

print(f"epsilon={eps:.2f}, AUROC={auroc:.3f}, TPR@1%FPR={tpr:.3f}, acc={acc:.1%}")

## 6. Bootstrap Confidence Intervals

Quantify uncertainty in your estimates using bootstrap resampling:

In [ ]:
# Configure bootstrap
params = BootstrapParams.confidence_interval(
    confidence=0.95,
    num_samples=1000,
    bias_correction=True,  # BCa bootstrap for better accuracy
    acceleration=True,
    seed=42,
)

print(f"Bootstrap config: {params}")

In [ ]:
# Bootstrap AUROC using the method-based API
result = AuditResult(in_scores_weak_dp, out_scores_weak_dp)

auroc_ci = result.bootstrap(AuditResult.auroc, params)

print(f"AUROC: {result.auroc():.3f}")
print(f"95% CI: [{auroc_ci[0]:.3f}, {auroc_ci[1]:.3f}]")

In [ ]:
# Bootstrap epsilon using a lambda
eps_ci = result.bootstrap(
    lambda r: r.epsilon_clopper_pearson(significance=0.05, delta=1e-5),
    params,
)

print(f"\nEpsilon: {result.epsilon_clopper_pearson(significance=0.05, delta=1e-5):.2f}")
print(f"95% CI: [{eps_ci[0]:.2f}, {eps_ci[1]:.2f}]")

## 7. Complete Auditing Workflow

Here's how to put it all together for a real DP model:

In [ ]:
def full_privacy_audit(in_scores, out_scores, theoretical_epsilon, delta=1e-5):
    """Run a complete privacy audit and compare to theoretical bounds."""
    
    # 1. Build audit result
    result = AuditResult(in_scores, out_scores)
    
    # 2. Compute confidence intervals
    params = BootstrapParams.confidence_interval(
        confidence=0.95, num_samples=1000, 
        bias_correction=True, acceleration=True, seed=42
    )
    
    eps_ci = result.bootstrap(
        lambda r: r.epsilon_clopper_pearson(significance=0.05, delta=delta),
        params,
    )
    auroc_ci = result.bootstrap(AuditResult.auroc, params)
    
    # 3. Report results
    print("=" * 50)
    print("PRIVACY AUDIT REPORT")
    print("=" * 50)
    print(f"\nSample sizes: {result.n_in} in, {result.n_out} out")
    print(f"Delta: {delta}")
    
    eps = result.epsilon_at(delta=delta, significance=0.05)
    
    print("\n--- Epsilon Bounds ---")
    print(f"Theoretical upper bound: {theoretical_epsilon:.2f}")
    print(f"Audited lower bound:     {eps:.2f} (95% CI: [{eps_ci[0]:.2f}, {eps_ci[1]:.2f}])")
    print(f"Gap: {theoretical_epsilon - eps:.2f}")
    
    print("\n--- Attack Metrics ---")
    print(f"AUROC:         {result.auroc():.3f} (95% CI: [{auroc_ci[0]:.3f}, {auroc_ci[1]:.3f}])")
    print(f"TPR at 1% FPR: {result.tpr_at_fpr(fpr=0.01):.3f}")
    print(f"Max accuracy:  {result.max_accuracy():.1%}")
    
    # 4. Sanity check
    print("\n--- Validation ---")
    if eps > theoretical_epsilon:
        print("WARNING: Audited epsilon exceeds theoretical bound!")
        print("    This may indicate a bug in your DP implementation.")
    else:
        print("OK: Audited epsilon is below theoretical bound (as expected)")
    
    return result

In [ ]:
# Run full audit on "Weak DP" scenario
# Assume theoretical epsilon was 8.0
_ = full_privacy_audit(
    in_scores_weak_dp, 
    out_scores_weak_dp, 
    theoretical_epsilon=8.0,
    delta=1e-5
)

In [ ]:
# Run full audit on "Good DP" scenario
# Assume theoretical epsilon was 3.0
_ = full_privacy_audit(
    in_scores_good_dp, 
    out_scores_good_dp, 
    theoretical_epsilon=3.0,
    delta=1e-5
)

## 8. Comparing Epsilon Methods

When should you use each method?

In [ ]:
# Compare methods across different sample sizes
sample_sizes = [50, 100, 200, 500, 1000]

results_cp = []
results_one = []

# Use weak DP scenario as base
in_full = np.random.normal(loc=0.7, scale=0.25, size=2000)
out_full = np.random.normal(loc=0.3, scale=0.25, size=2000)

for n in sample_sizes:
    r = AuditResult(in_full[:n], out_full[:n])
    results_cp.append(r.epsilon_clopper_pearson(significance=0.05))
    results_one.append(r.epsilon_one_run(significance=0.05))

# Plot
plt.figure(figsize=(10, 6))
plt.plot(sample_sizes, results_cp, marker='o', label="Clopper-Pearson")
plt.plot(sample_sizes, results_one, marker='s', label="One-run")

plt.xlabel("Sample Size (canaries per group)")
plt.ylabel("Estimated Epsilon")
plt.title("Epsilon Estimation Methods vs Sample Size")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nRecommendation:")
print("- Clopper-Pearson: Default choice, formal statistical guarantees")
print("- One-run: Less conservative with small samples, tighter for coin-flip setups")

## Summary

Key takeaways:

1. **Privacy auditing validates DP implementations** by empirically measuring attack success

2. **Two-line API for one-run auditing**:
   ```python
   import opaque.auditing as auditing
   experiment = auditing.setup(dataset, num_canaries=1000, seed=42)
   # ... train on experiment.subset(dataset) ...
   audit = auditing.evaluate(experiment, loss_fn, params, dataset)
   ```

3. **Epsilon estimation** via `result.epsilon_at(delta=)`:
   - Auto-selects method based on experiment type
   - `method="clopper_pearson"`: Conservative, general
   - `method="one_run"`: Tighter, for coin-flip setups

4. **Attack metrics** as methods on `AuditResult`:
   - `auroc()`: Overall attack strength
   - `tpr_at_fpr(fpr=)`: Worst-case performance
   - `max_accuracy()`: Easy to interpret

5. **Always report confidence intervals** using `result.bootstrap()`

6. **Audited epsilon should be <= theoretical epsilon**. If not, investigate your implementation!

## Next Steps

- **[Privacy Auditing User Guide](../user-guide/auditing.md)**: Detailed conceptual guide
- **[API Reference](../api/auditing.md)**: Full API documentation
- **Steinke, Nasr, Jagielski (2023)**: [Privacy Auditing with One (1) Training Run](https://arxiv.org/abs/2305.08846)